# 06 — Analyse d'erreur et recommandations

**Projet** : Moteur de recommandation de catalogue e-commerce avec scikit-learn
**Modèle analysé** : `hist_gradient_boosting` (Gradient boosting par histogrammes (HistGradientBoostingClassifier))
**Métrique de contrat** : `ndcg_at_k` (seuil `0.45`)

Une métrique moyenne ne dit jamais qui est mal servi. Ce notebook part du verdict produit par
l'évaluateur du projet, puis descend : par segment, par utilisateur, par emplacement publié. Il se
termine par des recommandations **vérifiables** — ce qu'on change, l'effet attendu, et la métrique
qui le prouve.

## Objectifs pédagogiques

1. Lire le verdict objectif par objectif, et savoir quoi faire de chacun.
1. Identifier les segments mal servis et le coût du démarrage froid.
1. Distinguer une instabilité réelle du bruit d'échantillonnage.
1. Remonter des pires classements à une cause actionnable.
1. Formuler des recommandations chiffrées et vérifiables.

**Objectifs transverses du dépôt**

- Construire un jeu de candidats (utilisateur x article) plutôt qu'une matrice d'interactions, et comprendre ce que ce choix autorise : consommer les features de fiche article, scorer un article neuf, publier sous contraintes métier.
- Formaliser le contrat d'antériorité de chaque feature : connue avant la session (fiche article, historique agrégé), connue au moment de la session (intention récente), ou interdite (issue de la session elle-même).
- Comprendre pourquoi l'unité d'évaluation est l'utilisateur et non la ligne : une précision calculée sur toutes les lignes mêle un client très actif à un nouveau venu et ne décrit aucun des deux.

## 0. Mise en place

La journalisation est descendue au niveau `ERROR` dans ce projet. Ce n'est pas un réglage de
confort : `fit` émet un avertissement par métrique `*_at_k` (`missing input(s) ['groups']`), parce
que la matrice pré-traitée ne porte plus l'identité de l'utilisateur et que le registre refuse, à
juste titre, de calculer une métrique **par utilisateur** sans groupe. Ces métriques sont
recalculées ici sur le cadre enrichi, et en production par le `Trainer` qui reçoit `groups_val`.
Les avertissements n'apporteraient donc rien — et noieraient les tableaux.

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet configure loguru au premier `get_logger()` appelé par `src`. On prend la main ici,
# au niveau ERROR : sans cela, chaque cellule d'entraînement noierait ses tableaux sous
# les lignes INFO de production. Les erreurs réelles restent visibles — c'est l'essentiel.
setup_logging(level="ERROR")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (6000 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 6000

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=ERROR",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.45)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

In [ ]:
from src.data.loaders import DatasetSplitter, feature_target_split
from src.features.build_features import FeatureBuilder, select_feature_columns, split_by_dtype
from src.preprocessing.pipelines import PreprocessingPipeline


def prepare_matrices(frame: pd.DataFrame, config: Any) -> dict[str, Any]:
    """Reproduce what ``TrainPipeline`` does, on the notebook-sized dataset.

    La fonction reprend **exactement** l'enchaînement de production : split → feature
    engineering (appris sur train uniquement) → preprocessing (appris sur train uniquement).
    C'est ce qui rend les chiffres de ce notebook comparables à ceux de `make train`.

    Args:
        frame: Raw dataset.
        config: Validated application configuration.

    Returns:
        Mapping with splits, fitted objects and model-ready matrices.
    """
    target = config.data.target
    drop_columns = list(config.data.drop_columns)

    splitter = DatasetSplitter.from_config(config.model_dump(), seed=config.seed)
    splits = splitter.split(frame, target=target)

    builder = FeatureBuilder.from_config(config.model_dump(), target=target)
    if builder.recipes:
        builder.fit(splits.train)
    enriched = {
        "train": builder.transform(splits.train),
        "val": None if splits.val is None else builder.transform(splits.val),
        "test": builder.transform(splits.test),
    }

    train_frame = enriched["train"]
    feature_columns = select_feature_columns(train_frame, drop_columns=drop_columns, target=target)
    numeric, categorical = split_by_dtype(train_frame, feature_columns)
    explicit = config.preprocessing.model_dump().get("columns") or {}
    numeric = list(explicit.get("numeric") or numeric)
    categorical = list(explicit.get("categorical") or categorical)

    pipeline = PreprocessingPipeline(
        numeric_features=numeric,
        categorical_features=categorical,
        config=config.preprocessing.model_dump(),
        target=target,
    )
    X_train_frame, y_train = feature_target_split(train_frame, target, drop_columns)
    X_train = pipeline.fit_transform(X_train_frame, y_train)

    def project(split: pd.DataFrame | None) -> tuple[pd.DataFrame | None, Any]:
        if split is None:
            return None, None
        _, labels = feature_target_split(split, target, drop_columns)
        return pipeline.transform(split.loc[:, X_train_frame.columns]), labels

    X_val, y_val = project(enriched["val"])
    X_test, y_test = project(enriched["test"])

    return {
        "splits": splits,
        "enriched": enriched,
        "builder": builder,
        "pipeline": pipeline,
        "numeric": numeric,
        "categorical": categorical,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": list(pipeline.feature_names_out),
        # Colonnes de la matrice **avant** pré-traitement (donc avant one-hot). Indispensables dès
        # qu'un notebook ré-applique le pipeline à un nouveau cadre : sélectionner les colonnes de
        # `X_train` (après one-hot) sur un cadre enrichi lève un KeyError sur les modalités.
        "frame_columns": list(X_train_frame.columns),
    }


PREPARED = prepare_matrices(raw, CONFIG)
print("train :", PREPARED["X_train"].shape)
print("val   :", None if PREPARED["X_val"] is None else PREPARED["X_val"].shape)
print("test  :", PREPARED["X_test"].shape)
print(f"features livrées au modèle : {len(PREPARED['feature_names'])}")
PREPARED["X_train"].head()

In [ ]:
import pandas as pd


def discounts(size: int) -> np.ndarray:
    """Return the NDCG positional discounts 1 / log2(rank + 1).

    Le discount logarithmique est ce qui distingue le NDCG d'une simple précision : être pertinent
    en première position vaut plus qu'en dixième, parce que l'attention de l'utilisateur décroît
    avec la position. log2(rank + 1) est la forme standard, retenue parce qu'elle décroît vite au
    début de la liste et lentement ensuite — exactement le profil d'attention observé.
    """
    if size <= 0:
        return np.zeros(0)
    return 1.0 / np.log2(np.arange(2, size + 2, dtype="float64"))


def dcg(gains: np.ndarray) -> float:
    """Return the discounted cumulative gain of an ordered gain vector."""
    gains = np.asarray(gains, dtype="float64")
    return float((gains * discounts(gains.size)).sum()) if gains.size else 0.0


def ndcg_at_k(truth: np.ndarray, score: np.ndarray, k: int) -> float:
    """Return the NDCG@K of one list: realised gain over the best possible gain.

    Retourner 0 quand la liste ne contient rien de pertinent est un choix : un utilisateur sans
    intention ne peut pas être satisfait, et compter sa liste comme « parfaite » récompenserait un
    moteur qui publie peu. Ces utilisateurs sont donc exclus du calcul par l'appelant.
    """
    truth = np.asarray(truth, dtype="float64")
    order = np.argsort(-np.asarray(score, dtype="float64"), kind="stable")
    ideal = dcg(np.sort(truth)[::-1][:k])
    return dcg(truth[order][:k]) / ideal if ideal > 0 else 0.0


def precision_at_k(truth: np.ndarray, score: np.ndarray, k: int) -> float:
    """Return the share of the K published slots that are relevant."""
    order = np.argsort(-np.asarray(score, dtype="float64"), kind="stable")
    selected = np.asarray(truth, dtype="float64")[order][:k]
    return float(selected.sum() / selected.size) if selected.size else 0.0


def recall_at_k(truth: np.ndarray, score: np.ndarray, k: int) -> float:
    """Return the share of the available relevance captured by the K published slots."""
    truth = np.asarray(truth, dtype="float64")
    total = float(truth.sum())
    if total <= 0:
        return float("nan")
    order = np.argsort(-np.asarray(score, dtype="float64"), kind="stable")
    return float(truth[order][:k].sum() / total)


def average_precision_at_k(truth: np.ndarray, score: np.ndarray, k: int) -> float:
    """Return AP@K: the mean of the precisions measured at each relevant hit."""
    order = np.argsort(-np.asarray(score, dtype="float64"), kind="stable")
    selected = np.asarray(truth, dtype="float64")[order][:k]
    if selected.sum() <= 0:
        return 0.0
    precisions = np.cumsum(selected) / np.arange(1, selected.size + 1, dtype="float64")
    return float((precisions * selected).sum() / min(selected.sum(), k))


def hit_rate_at_k(truth: np.ndarray, score: np.ndarray, k: int) -> float:
    """Return 1 when at least one relevant item lands in the K published slots."""
    order = np.argsort(-np.asarray(score, dtype="float64"), kind="stable")
    return 1.0 if float(np.asarray(truth, dtype="float64")[order][:k].sum()) > 0 else 0.0


def ranking_metrics(frame: pd.DataFrame, score_column: str, k: int = 10) -> pd.Series:
    """Compute every ranking metric **per user**, then average.

    C'est la fonction la plus importante du notebook : elle matérialise le fait que l'unité
    d'évaluation est l'utilisateur. Les utilisateurs sans aucun candidat pertinent sont exclus,
    parce qu'aucun classement ne peut les satisfaire.
    """
    rows = {"ndcg": [], "precision": [], "recall": [], "map": [], "hit": []}
    for _, group in frame.groupby("user_id", observed=True, sort=False):
        truth = group[TARGET].to_numpy(dtype="float64")
        if truth.sum() <= 0:
            continue
        score = group[score_column].to_numpy(dtype="float64")
        rows["ndcg"].append(ndcg_at_k(truth, score, k))
        rows["precision"].append(precision_at_k(truth, score, k))
        rows["recall"].append(recall_at_k(truth, score, k))
        rows["map"].append(average_precision_at_k(truth, score, k))
        rows["hit"].append(hit_rate_at_k(truth, score, k))
    if not rows["ndcg"]:
        return pd.Series(dtype="float64")
    values = pd.Series({key: float(np.nanmean(item)) for key, item in rows.items()})
    values["utilisateurs"] = float(len(rows["ndcg"]))
    return values.round(4)


def score_frame(PREPARED: dict[str, Any], split: str, model: Any) -> pd.DataFrame:
    """Return the enriched split rows with the model score, ready for per-user metrics.

    Le cadre enrichi (avant pré-traitement) porte les identifiants et les colonnes métier ; la
    matrice pré-traitée porte ce que le modèle consomme. Les deux sont alignés ligne à ligne, ce
    qui permet de scorer sur l'une et d'analyser sur l'autre.
    """
    frame = PREPARED["enriched"][split].reset_index(drop=True)
    matrix = PREPARED[f"X_{split}"]
    probabilities = model.predict_proba(matrix)
    scored = frame.copy()
    scored["score"] = probabilities[:, -1] if probabilities.ndim == 2 else probabilities
    return scored

In [ ]:
from src.models import build_model

MODEL = build_model(CONFIG, feature_names=PREPARED["feature_names"])
# `build_model(params=...)` **remplace** les réglages de `conf/model/default.yaml`. Pour ne faire
# varier qu'un seul facteur à la fois dans les sections suivantes (algorithme, coupure, graine,
# grille), on part donc toujours de cette copie des réglages configurés.
CONFIGURED_PARAMS = dict(CONFIG.model.params)
FIT_RESULT = MODEL.fit(
    PREPARED["X_train"],
    PREPARED["y_train"],
    X_val=PREPARED["X_val"],
    y_val=PREPARED["y_val"],
    callbacks=[],
)
print(MODEL.summary())
print(f"entraînement : {FIT_RESULT.duration_seconds:.2f} s")

In [ ]:
# Aucun nom de colonne n'est écrit dans ce notebook : l'évaluateur du projet résout ses réglages
# depuis `conf/config.yaml` (bloc `recommendation`, avec repli sur le noeud `data`). Le notebook
# utilise exactement les mêmes colonnes que le rapport de production — changer le schéma ne
# demande qu'un changement de configuration, pas une édition de notebook.
from src.evaluation.evaluator import RankingSettings

CONFIG_DICT = CONFIG.model_dump(mode="json")
SETTINGS = RankingSettings.resolve(CONFIG_DICT)


def rank_scores(model: Any, matrix: pd.DataFrame) -> np.ndarray:
    """Return a continuous ranking score, whatever the estimator exposes.

    Un score de classement n'a pas besoin d'être une probabilité : il lui faut seulement être
    **ordonnable**. Un SVM sans calibration probabiliste classe très bien par sa distance à la
    frontière, un arbre par sa probabilité, un modèle de factorisation par son produit scalaire.
    Cette fonction prend ce que l'estimateur sait produire, dans l'ordre de préférence.
    """
    try:
        probabilities = np.asarray(model.predict_proba(matrix), dtype="float64")
        return probabilities[:, -1].ravel() if probabilities.ndim == 2 else probabilities.ravel()
    except Exception:
        estimator = getattr(model, "estimator_", None) or getattr(model, "model_", None)
        for attribute in ("decision_function", "score_samples", "predict"):
            function = getattr(estimator, attribute, None)
            if callable(function):
                values = np.asarray(function(matrix), dtype="float64")
                return values[:, -1].ravel() if values.ndim == 2 else values.ravel()
        raise


VAL = PREPARED["enriched"]["val"].reset_index(drop=True).copy()
VAL["score"] = rank_scores(MODEL, PREPARED["X_val"])
TARGET = CONFIG.data.target
GROUPE = SETTINGS.group_column
ARTICLE = SETTINGS.item_column
TOP_K = int(SETTINGS.top_k)
print(f"split de validation : {len(VAL):,} candidats, {VAL[GROUPE].nunique()} utilisateurs")
print(f"coupure publiée     : top-{TOP_K}")
print(
    f"colonnes résolues   : groupe={GROUPE} article={ARTICLE} "
    f"popularité={SETTINGS.popularity_column} intention={SETTINGS.intent_column}"
)

## 1. Le verdict d'abord

L'évaluateur du projet calcule les métriques par utilisateur, les références, la couverture, le
backtest chronologique, les pires classements et l'importance des features. Le notebook ne recalcule
rien : il interprète.

In [ ]:
from src.evaluation.evaluator import Evaluator

# L'évaluateur du projet fait tout le travail de mesure : il calcule les métriques par
# utilisateur, les références, la couverture, le backtest chronologique, les pires classements et
# l'importance des features. Le notebook ne recalcule rien — il **interprète** et prolonge là où
# le rapport écrit ne peut pas aller (lire des lignes, tester une hypothèse, chiffrer un arbitrage).
# `Evaluator.from_config` attend un **mapping** (comme le pipeline `scripts/evaluate.py`, qui
# passe le dictionnaire Hydra). `CONFIG` est le modèle pydantic validé : on le sérialise, ce qui
# inclut le bloc `recommendation` déclaré dans `conf/config.yaml`.
evaluator = Evaluator.from_config(MODEL, CONFIG.model_dump(mode="json"), PATHS)
RESULT = evaluator.evaluate(
    PREPARED["X_test"],
    PREPARED["y_test"],
    split="test",
    context=PREPARED["enriched"]["test"],
)
VERDICTS = RESULT.extras["verdicts"]

print(f"split        : {RESULT.split} | {RESULT.n_samples:,} candidats")
print(
    f"utilisateurs : {RESULT.extras['users_evaluated']} évalués sur {RESULT.extras['users_total']}"
)
print(f"métrique clé : {RESULT.primary_metric} = {RESULT.primary_value:.4f}")
print(f"statut       : {VERDICTS['statut']}")

In [ ]:
# Chaque objectif est une phrase du contrat de service, pas un nombre décoratif. Le tableau dit
# ce qui est tenu, ce qui ne l'est pas, et surtout **pourquoi** l'objectif existe.
# `VERDICTS` a la forme produite par l'évaluateur : une liste d'objectifs sous la clé
# `objectifs`, et le décompte global sous `statut` / `respectes` / `mesurables`.
entries = list(VERDICTS.get("objectifs") or [])
if entries:
    objectifs = pd.DataFrame(entries).set_index("critere")
    display(objectifs[["mesure", "seuil", "operateur", "verdict"]])
    print()
    print(
        f"{VERDICTS.get('respectes')}/{VERDICTS.get('mesurables')} objectifs mesurables "
        f"respectés — statut : {VERDICTS.get('statut')}"
    )
    print()
    for entry in entries:
        marque = {"respecte": "OK ", "non_respecte": "KO ", "non_mesurable": "-- "}.get(
            str(entry.get("verdict")), "?? "
        )
        print(f"  {marque}{entry.get('critere')} : {entry.get('lecture')}")
else:  # forme plate : {nom: booléen}
    objectifs = pd.DataFrame(
        [
            {"objectif": name, "statut": "acquis" if ok else "manqué"}
            for name, ok in VERDICTS.items()
        ]
    )
    display(objectifs)
    print(f"{int((objectifs['statut'] == 'acquis').sum())}/{len(objectifs)} objectifs acquis")
print()
print("Lecture : un objectif manqué n'est pas une note, c'est une décision à prendre.")
print("« NDCG sous le seuil » → travailler les features ou le modèle.")
print("« couverture sous le seuil » → contrainte de diversité à publier, pas un bug de score.")
print("« rappel froid sous le seuil » → le moteur ne sert pas les nouveaux : problème produit.")

**Ce qu'il faut retenir**

- Un objectif manqué n'est pas une note, c'est une décision à prendre — et chaque objectif pointe vers un chantier différent.
- Le statut global (`conforme`, `partiel`, `non_conforme`) est contractuel : il conditionne la mise en production, pas la discussion technique.

## 2. Ventilation par segments

La moyenne cache toujours un segment. Les segments affichés correspondent à des décisions :
démarrage froid (produit), catégorie (catalogue), période (saisonnalité).

In [ ]:
# La moyenne cache toujours un segment. Ici les segments sont ceux qui correspondent à des
# décisions : le démarrage froid (produit), la catégorie (catalogue), la période (saisonnalité).
segments = RESULT.per_segment
if segments.empty:
    print("Aucune ventilation produite (colonnes de segmentation absentes du contexte).")
else:
    # L'index brut est un numéro de ligne : sans libellé, `idxmax()` renverrait « 1 » au lieu de
    # « segment_utilisateur / chaud ». On construit donc un index lisible avant tout classement.
    if {"axe", "segment"}.issubset(segments.columns):
        segments = segments.set_index(
            segments["axe"].astype(str) + " / " + segments["segment"].astype(str)
        )
    display(segments)

    print()
    print("Ce que la ventilation dit de façon exploitable :")
    numerique = segments.select_dtypes("number")
    if not numerique.empty and "ndcg" in numerique.columns:
        meilleur = numerique["ndcg"].idxmax()
        pire = numerique["ndcg"].idxmin()
        ecart = float(numerique.loc[meilleur, "ndcg"] - numerique.loc[pire, "ndcg"])
        print(f"  segment le mieux servi : {meilleur} (NDCG {numerique.loc[meilleur, 'ndcg']:.4f})")
        print(f"  segment le moins servi : {pire} (NDCG {numerique.loc[pire, 'ndcg']:.4f})")
        print(f"  écart                  : {ecart:+.4f}")
        print()
        print("Un écart important entre segments justifie un modèle ou un réglage par segment,")
        print("ou au minimum un objectif de service distinct — pas une moyenne unique.")

    if not RESULT.baselines.empty:
        print()
        print("Références mesurées sur le même split :")
        display(RESULT.baselines)

**Ce qu'il faut retenir**

- Un écart important entre segments justifie un réglage par segment, ou au minimum un objectif de service distinct — jamais une moyenne unique.
- Les références mesurées sur le même split permettent de savoir si un segment faible l'est aussi pour la popularité : si oui, le problème vient de la donnée, pas du modèle.

## 3. Le démarrage froid

C'est le point de rupture d'un moteur de recommandation : la référence « intention récente » y
tombe à zéro, et le catalogue populaire ne dit rien de la personne. On lit donc ce que le moteur
publie **réellement** pour ces utilisateurs.

In [ ]:
# Le démarrage froid est LE point de rupture d'un moteur de recommandation : c'est là que la
# référence « intention récente » tombe à zéro (pas d'historique) et que le catalogue populaire
# ne dit rien de la personne. On lit donc ce que le moteur publie réellement pour ces utilisateurs.
ratio_froid = RESULT.extras.get("cold_start_recall_ratio", float("nan"))
print(f"ratio de rappel froid / global : {ratio_froid:.2f}")
print("1.00 = les nouveaux sont servis exactement comme les habitués")
print("<0.80 = le moteur dégrade systématiquement le service des nouveaux arrivants")
print()

published = RESULT.predictions
if published.empty:
    print("Aucune publication produite.")
else:
    colonne_activite = evaluator.settings.activity_column
    colonne_groupe = evaluator.settings.group_column
    # Le cadre publié ne porte que les colonnes nécessaires à la recommandation : l'activité de
    # l'utilisateur reste dans le cadre enrichi du split. On la recolle par identifiant, ce qui
    # évite de supposer que les deux cadres partagent leurs colonnes.
    contexte = PREPARED["enriched"]["test"]
    if colonne_activite not in published.columns and colonne_activite in contexte.columns:
        activite = contexte.groupby(colonne_groupe, observed=True)[colonne_activite].first()
        published = published.assign(**{colonne_activite: published[colonne_groupe].map(activite)})
    if colonne_activite in published.columns:
        froid = published[published[colonne_activite] <= evaluator.settings.cold_user_max_orders]
        chaud = published[published[colonne_activite] > evaluator.settings.cold_user_max_orders]
        comparaison = pd.DataFrame(
            {
                "utilisateurs": pd.Series(
                    {
                        "démarrage froid": froid[colonne_groupe].nunique(),
                        "habitués": chaud[colonne_groupe].nunique(),
                    }
                ),
                "publications": pd.Series({"démarrage froid": len(froid), "habitués": len(chaud)}),
                "score moyen": pd.Series(
                    {
                        "démarrage froid": froid["y_pred"].mean()
                        if "y_pred" in froid
                        else float("nan"),
                        "habitués": chaud["y_pred"].mean() if "y_pred" in chaud else float("nan"),
                    }
                ),
                "articles distincts": pd.Series(
                    {
                        "démarrage froid": froid[evaluator.settings.item_column].nunique(),
                        "habitués": chaud[evaluator.settings.item_column].nunique(),
                    }
                ),
            }
        )
        display(comparaison)

        print()
        print("Diagnostic : pour un utilisateur froid, le moteur ne peut s'appuyer que sur les")
        print("features **article** (popularité, note, prix, marge) et **contexte** (période,")
        print("canal). S'il publie exactement le même top-K pour tous les nouveaux, c'est que")
        print("ces features dominent — et la personnalisation n'existe pas encore pour eux.")
        if froid[colonne_groupe].nunique() > 0:
            premiers = (
                froid.groupby(colonne_groupe, observed=True)[evaluator.settings.item_column]
                .apply(lambda items: tuple(items[:5]))
                .head(6)
            )
            print()
            print("Top-5 publié pour quelques utilisateurs froids :")
            display(premiers.rename("top-5").to_frame())
            distincts = len(set(premiers.tolist()))
            print(f"{distincts} listes distinctes sur {premiers.size} utilisateurs froids")
            if distincts == 1:
                print(
                    "=> liste identique pour tous : le moteur est en mode « catalogue populaire »."
                )
            else:
                print(
                    "=> les listes diffèrent : le contexte et l'article produisent de la variation."
                )
    else:
        print(f"Colonne d'activité '{colonne_activite}' absente des publications :")
        print("le segment froid ne peut pas être isolé dans ce notebook.")

**Ce qu'il faut retenir**

- Un ratio de rappel froid proche de 1 signifie que les nouveaux sont servis comme les habitués ; nettement sous 1, le moteur les dégrade systématiquement.
- Si la liste publiée est identique pour tous les nouveaux, le moteur est en mode « catalogue populaire » : la personnalisation n'existe pas encore pour eux.
- Pour un utilisateur froid, seules les features article et contexte portent de l'information — d'où l'intérêt de la note, de la nouveauté et de la saisonnalité.

## 4. Couverture, concentration et biais de popularité

Un moteur pertinent mais monotone est un moteur rejeté : il épuise le catalogue, lasse les
utilisateurs et négocie mal avec les fournisseurs. Ce sont des contraintes de service, pas des
métriques de confort.

In [ ]:
# Un moteur pertinent mais monotone est un moteur rejeté : il épuise le catalogue, lasse les
# utilisateurs et négocie mal avec les fournisseurs. La couverture et la concentration ne sont pas
# des métriques de confort, ce sont des contraintes de service.
couverture = RESULT.extras.get("catalog_coverage", float("nan"))
concentration = RESULT.extras.get("herfindahl_top_k", float("nan"))
biais_popularite = RESULT.extras.get("popularity_bias_share", float("nan"))
rupture = RESULT.extras.get("out_of_stock_published_share", float("nan"))

table = pd.Series(
    {
        "couverture catalogue (top-K union)": couverture,
        "minimum exigé": evaluator.settings.coverage_min,
        "Herfindahl des publications": concentration,
        "part des slots prise par le décile chaud": biais_popularite,
        "publications en rupture de stock": rupture,
    },
    name="valeur",
).to_frame()
display(table)

print()
print("Lecture :")
print(
    f"  {couverture:.1%} du catalogue apparaît dans au moins un top-K "
    f"(minimum exigé {evaluator.settings.coverage_min:.0%})"
)
if biais_popularite == biais_popularite:
    print(f"  {biais_popularite:.1%} des emplacements reviennent au décile d'articles le plus vu :")
    print("  c'est la mesure directe du biais de popularité du moteur.")
if rupture == rupture:
    print(f"  {rupture:.1%} des publications concernent un article en rupture : gaspillage")
    print("  d'emplacement, corrigé à l'inférence par le filtre de disponibilité.")

if not published.empty:
    colonne_article = evaluator.settings.item_column
    compte = published[colonne_article].value_counts()
    parts = compte.to_numpy(dtype="float64") / float(compte.sum())
    courbe = pd.DataFrame(
        {
            "part du catalogue": [0.01, 0.05, 0.10, 0.25, 0.50],
        }
    )
    cumulees = []
    for part in courbe["part du catalogue"]:
        n = max(1, round(part * compte.size))
        cumulees.append(round(float(parts[:n].sum()), 4))
    courbe["part des publications"] = cumulees
    display(courbe)
    print()
    print("Si 1 % du catalogue capte une part disproportionnée des publications, la")
    print("diversité est le prochain chantier — avant toute amélioration du NDCG.")

**Ce qu'il faut retenir**

- La part des emplacements captée par le décile d'articles le plus vu est la mesure directe du biais de popularité du moteur.
- Les publications en rupture de stock sont des emplacements perdus : elles se corrigent à l'inférence par un filtre de disponibilité, pas par le score.
- Si 1 % du catalogue capte une part disproportionnée des publications, la diversité est le chantier suivant — avant toute amélioration du NDCG.

## 5. Backtest et plancher de bruit

Un NDCG moyen sur toute la période cache les saisons. La question décisive n'est pas « ça varie ? »
— ça varie toujours — mais « ça varie **plus que le bruit** d'échantillonnage d'un pli ? ».

In [ ]:
# Un NDCG moyen sur toute la période cache les saisons. Le backtest découpe le test en plis
# chronologiques et mesure la dispersion. La question décisive n'est pas « ça varie ? » — ça varie
# toujours — mais « ça varie plus que le bruit d'échantillonnage ? ».
temporel = RESULT.backtest
if temporel.empty:
    print("Backtest non produit (colonne temporelle absente du contexte).")
else:
    display(temporel)
    dispersion = RESULT.extras.get("dispersion_backtest", {})
    if dispersion:
        display(pd.Series(dispersion, name="dispersion entre plis").to_frame())
        amplitude = float(dispersion.get("amplitude", float("nan")))
        plafond_bruit = float(
            dispersion.get("plancher_bruit", dispersion.get("bruit", float("nan")))
        )
        seuil = evaluator.settings.backtest_spread_max
        print()
        print(f"amplitude observée   : {amplitude:.4f}")
        print(f"seuil contractuel    : {seuil:.4f}")
        if plafond_bruit == plafond_bruit:
            print(f"plancher de bruit    : {plafond_bruit:.4f}")
        print()
        if amplitude <= seuil:
            print("Sous le seuil : la qualité est stable d'une période à l'autre.")
            print("ATTENTION à ne pas lire « stable » comme « bon » : un moteur médiocre peut")
            print(
                "être très régulièrement médiocre. Le backtest mesure la dispersion, pas le niveau."
            )
        else:
            print("Au-dessus du seuil : chercher une cause (saisonnalité, dérive du catalogue,")
            print("vieillissement des features) avant de retoucher les hyperparamètres.")

    colonne_temps = evaluator.settings.time_column
    if colonne_temps and colonne_temps in PREPARED["enriched"]["test"].columns:
        test = PREPARED["enriched"]["test"]
        periode = test.groupby(
            pd.to_datetime(test[colonne_temps]).dt.to_period("M"), observed=True
        ).size()
        print()
        print("Volume de candidats par mois dans le test (le contexte du backtest) :")
        display(periode.rename("candidats").to_frame())

**Ce qu'il faut retenir**

- Le seuil contractuel d'amplitude est calibré au-dessus du plancher de bruit : en dessous de ce plancher, le critère mesurerait le hasard.
- « Stable » ne veut pas dire « bon » : un moteur médiocre peut être très régulièrement médiocre. Le backtest mesure la dispersion, pas le niveau.
- Une amplitude qui dépasse le seuil appelle une cause (saisonnalité, dérive du catalogue, vieillissement des features) avant tout retouchage d'hyperparamètres.

## 6. Les pires classements

C'est la matière première du diagnostic : on y lit ce que le moteur a publié, ce qu'il aurait dû
publier, et quelle information lui a manqué.

In [ ]:
# Les pires classements sont la matière première du diagnostic : on y lit ce que le moteur a
# publié, ce qu'il aurait dû publier, et quelle feature l'a trompé.
erreurs = RESULT.errors
if erreurs.empty:
    print("Aucun mauvais classement produit par l'évaluateur.")
else:
    display(erreurs.head(12))
    print()
    print("Colonnes utiles pour le diagnostic :")
    for colonne in erreurs.columns:
        print(f"  {colonne}")
    print()
    colonne_groupe = evaluator.settings.group_column
    if colonne_groupe in erreurs.columns:
        pires = erreurs[colonne_groupe].value_counts().head(5)
        print("Utilisateurs les plus mal servis :")
        display(pires.rename("occurrences").to_frame())
        print()
        print("Un utilisateur qui revient plusieurs fois dans les pires classements n'est pas")
        print("un accident : c'est un profil que les features ne décrivent pas. C'est le")
        print("signal d'une feature manquante (intention, saison, budget), pas d'un réglage.")

**Ce qu'il faut retenir**

- Un utilisateur qui revient plusieurs fois dans les pires classements n'est pas un accident : c'est un profil que les features ne décrivent pas.
- Le signal appelle une feature manquante (intention, saison, budget), pas un réglage de capacité.

## 7. Importance par permutation

La seule question qui compte pour un classement : si je détruis l'information de cette colonne, de
combien la métrique tombe-t-elle ? Une feature importante pour prédire la ligne peut être inutile
pour ordonner la liste — et inversement.

In [ ]:
# L'importance par permutation répond à la seule question qui compte pour un classement : si je
# détruis l'information de cette colonne, de combien le NDCG tombe-t-il ? Une feature importante
# pour prédire la ligne peut être inutile pour ordonner la liste — et inversement.
importance = RESULT.feature_importance
if importance.empty:
    print("Aucune importance produite (le modèle n'expose pas de scores exploitables).")
else:
    display(importance)

    colonne_delta = next(
        (
            name
            for name in importance.columns
            if any(
                marque in name.lower()
                for marque in ("importance", "delta", "perte", "drop", "cout", "coût")
            )
        ),
        None,
    )
    if colonne_delta is not None:
        classement = importance.sort_values(colonne_delta, ascending=False)
        print()
        print(
            f"Classement par {colonne_delta} (chute de la métrique quand la colonne est mélangée) :"
        )
        display(classement.head(10))
        print()
        premier = classement.iloc[0]
        print(f"Feature dominante : {premier.name}")
        print("Si une seule feature porte l'essentiel du classement, le moteur est fragile :")
        print("sa qualité disparaît avec la disponibilité de cette colonne.")
    else:
        print()
        print("Colonnes produites : " + ", ".join(importance.columns))

    print()
    print("Mise en garde : une colonne de popularité dominante est attendue — c'est la")
    print("référence à battre. Ce qui doit inquiéter, c'est une colonne qui reproduirait la")
    print("cible (fuite) ou une colonne d'identité utilisateur (mémorisation sans généralisation).")

**Ce qu'il faut retenir**

- Une colonne de popularité dominante est attendue : c'est la référence à battre, pas une anomalie.
- Ce qui doit inquiéter, c'est une colonne qui reproduirait la cible (fuite) ou une identité utilisateur (mémorisation sans généralisation).
- Si une seule feature porte l'essentiel du classement, le moteur est fragile : sa qualité disparaît avec la disponibilité de cette colonne.

## 8. Hypothèses et recommandations

Chaque recommandation est une action vérifiable : ce qu'on change, l'effet attendu, la métrique qui
le prouve. Une recommandation sans métrique cible est une opinion.

In [ ]:
# Recommandations. Chacune est formulée comme une action vérifiable : ce qu'on change, l'effet
# attendu sur quelle métrique, et comment on le mesure. Une recommandation sans métrique cible
# est une opinion.
recommandations = [
    {
        "n°": 1,
        "action": "Filtrer les ruptures de stock APRÈS le classement et republier le candidat "
        "suivant (déjà implémenté dans `src/inference/predictor.py`).",
        "effet attendu": "supprime les emplacements perdus sans toucher au NDCG mesuré",
        "comment vérifier": "part des publications en rupture = 0 sur le test d'inférence",
    },
    {
        "n°": 2,
        "action": "Ajouter des features d'intention de court terme (sessions en cours, panier, "
        "recherches) plutôt que d'augmenter la capacité du modèle.",
        "effet attendu": "NDCG et rappel, surtout sur le segment chaud",
        "comment vérifier": "gain de NDCG supérieur à la dispersion entre graines (notebook 04)",
    },
    {
        "n°": 3,
        "action": "Publier un top-K dédié au démarrage froid : tri par popularité pondérée par la "
        "note et la nouveauté, faute d'historique.",
        "effet attendu": "ratio de rappel froid vers 1.00",
        "comment vérifier": "ratio rappel froid / global, par segment, sur le backtest",
    },
    {
        "n°": 4,
        "action": "Introduire une contrainte de diversité (max par catégorie) dans la publication.",
        "effet attendu": "couverture et Herfindahl, au prix d'un léger recul du NDCG",
        "comment vérifier": "table de sensibilité à K et arbitrage marge du notebook 04",
    },
    {
        "n°": 5,
        "action": "Arbitrer explicitement pertinence contre marge dans le score de publication, "
        "avec un poids configurable.",
        "effet attendu": "valeur par publication, NDCG en léger recul",
        "comment vérifier": "courbe marge/NDCG du rapport, point de fonctionnement retenu",
    },
    {
        "n°": 6,
        "action": "Rejouer le backtest à chaque réentraînement et comparer l'amplitude au plancher "
        "de bruit, jamais au seul seuil contractuel.",
        "effet attendu": "détection des dérives saisonnières avant qu'elles soient visibles",
        "comment vérifier": "amplitude inter-plis et son erreur type dans le rapport",
    },
]
display(pd.DataFrame(recommandations).set_index("n°"))

print()
print("Ce que ce notebook NE recommande PAS, volontairement :")
print("  - augmenter la profondeur ou le nombre d'arbres : la grille du notebook 04 montre")
print("    que le gain reste sous la dispersion entre graines ;")
print("  - évaluer en AUC ou en précision globale : ces métriques mélangent des utilisateurs")
print("    de volumes différents et ne décrivent pas le service rendu ;")
print("  - choisir le K sur le NDCG seul : le K est un arbitrage qualité/diversité/coût.")

**Ce qu'il faut retenir**

- Les recommandations portent d'abord sur la donnée et la publication (features d'intention, filtre de stock, top-K froid, diversité) : c'est là que le gain dépasse la dispersion entre graines.
- Augmenter la capacité du modèle est explicitement écarté tant que le gain reste sous le bruit mesuré au notebook 04.
- Le choix de K et l'arbitrage marge sont des décisions produit : elles se prennent sur des courbes, pas sur une métrique unique.

## 9. Figures du rapport

In [ ]:
# Les figures du rapport : elles servent à trancher une décision, pas à illustrer le code. Le
# générateur est celui de la production (`RankingPlots`, utilisé par `src/evaluation/reports.py`),
# et les fichiers sont écrits dans `outputs/notebooks` — jamais dans `artifacts/`, que
# `make evaluate` possède.
from src.visualization.plots import RankingPlots

figures = NB_PATHS.figures_dir
try:
    produites = RankingPlots(figures).save_all(RESULT)
    print(f"{len(produites)} figures produites dans {figures}")
    for nom, chemin in sorted(produites.items()):
        print(f"  - {nom:<26} {chemin.name}")
except Exception as error:  # une figure manquante ne doit pas masquer l'analyse
    produites = {}
    print(f"Figures non produites ({type(error).__name__}: {error}).")
    print("Le rapport écrit (`scripts/evaluate.py`) reste la source de vérité.")

# Deux figures suffisent à résumer le verdict : la qualité contre ses seuils, et la position du
# modèle entre le plancher et le plafond.
for cle in ("metrics_bar", "ceiling_position"):
    chemin = produites.get(cle)
    if chemin is not None and Path(chemin).exists():
        display(Image(filename=str(chemin)))

## Conclusion

Le verdict a été lu objectif par objectif, les segments mal servis ont été identifiés, le démarrage
froid a été mesuré au niveau des publications réelles, la couverture et le biais de popularité ont
été chiffrés, et l'instabilité a été comparée à son plancher de bruit. Les recommandations qui en
découlent sont actionnables et vérifiables.

Ce qui distingue cette analyse d'un rapport de classification : l'unité n'est jamais la ligne,
toujours l'utilisateur ; et le critère de succès n'est jamais la métrique seule, toujours la
métrique **avec** ses contraintes de service (couverture, disponibilité, équité entre segments).